# 06 — Agente explicativo del forecast

Capa GenAI sobre el forecast. Arquitectura **3 roles** inspirada en smolagents/CrewAI:

1. **DataAgent** — lee el CSV cacheado y resume la serie.
2. **ForecastAgent** — carga el forecast guardado por NB 02/04/05 y extrae claves cuantitativas.
3. **ReporterAgent** — redacta en castellano un informe ejecutivo accionable.

> **Nota sobre el stack:** este notebook trae una implementación **template-based** (sin LLM) que funciona offline y es determinista para evaluación reproducible. La actualización a smolagents/CrewAI/FinGPT (módulo 10) sólo requiere sustituir las funciones de los tres agentes manteniendo la misma firma.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
from datetime import datetime

from src.data_loader import OPA_BBVA_EVENTS

## DataAgent — resumen del histórico

In [ ]:
def data_agent(path_clean=ROOT / 'data' / 'sab_5y_clean.csv'):
    sab = pd.read_csv(path_clean, parse_dates=['Date'])
    last = sab.iloc[-1]
    first = sab.iloc[0]
    return {
        'rango': f"{first['Date'].date()} -> {last['Date'].date()}",
        'sesiones': len(sab),
        'cierre_actual': round(float(last['Close']), 3),
        'cierre_max': round(float(sab['Close'].max()), 3),
        'cierre_min': round(float(sab['Close'].min()), 3),
        'retorno_5y_pct': round((last['Close']/first['Close']-1)*100, 1),
        'volatilidad_anualizada_pct': round(float(sab['Close'].pct_change().std()*np.sqrt(252)*100), 2),
    }

resumen_serie = data_agent()
resumen_serie

## ForecastAgent — lee la previsión del modelo elegido

In [ ]:
def forecast_agent(path_fcst=ROOT / 'data' / 'sab_forecast_prophet_90d.csv'):
    fcst = pd.read_csv(path_fcst, parse_dates=['ds'])
    fin = fcst.iloc[-1]
    cierre_hoy = float(data_agent()['cierre_actual'])
    return {
        'fuente': path_fcst.name,
        'horizonte_dias': len(fcst),
        'fecha_final': fin['ds'].date().isoformat(),
        'precio_esperado': round(float(fin['yhat']), 3),
        'ic80_inf': round(float(fin['yhat_lower']), 3),
        'ic80_sup': round(float(fin['yhat_upper']), 3),
        'variacion_pct': round((float(fin['yhat'])/cierre_hoy - 1) * 100, 2),
    }

pred = forecast_agent()
pred

## ReporterAgent — informe en castellano

Template-based con datos de los dos agentes anteriores + recordatorio de los hitos OPA relevantes.

In [ ]:
def reporter_agent(resumen_serie, pred, holidays=OPA_BBVA_EVENTS):
    var = pred['variacion_pct']
    if var > 5:    tendencia = 'al alza significativa'
    elif var > 1:  tendencia = 'ligeramente alcista'
    elif var > -1: tendencia = 'lateral'
    elif var > -5: tendencia = 'ligeramente bajista'
    else:          tendencia = 'a la baja significativa'

    hitos_recientes = holidays.sort_values('ds').tail(3)
    bullets_hitos = '\n'.join(f"  - {row['ds'].date()}: {row['holiday']}" for _, row in hitos_recientes.iterrows())

    informe = f"""# Informe de previsión — Banco Sabadell (SAB.MC)

*Generado por agente FinanzIA el {datetime.utcnow().date()}*

## Situación actual
El precio de cierre más reciente de SAB.MC es **{resumen_serie['cierre_actual']} EUR** sobre una serie que abarca {resumen_serie['sesiones']} sesiones bursátiles ({resumen_serie['rango']}). En los últimos 5 años el activo ha acumulado un retorno del **{resumen_serie['retorno_5y_pct']}%** con una volatilidad anualizada del **{resumen_serie['volatilidad_anualizada_pct']}%**.

## Previsión a {pred['horizonte_dias']} días ({pred['fuente']})
Nuestro modelo proyecta una cotización **{tendencia}** hacia la fecha **{pred['fecha_final']}**, con un precio esperado de **{pred['precio_esperado']} EUR** ({var:+.2f}% sobre el cierre actual). El intervalo de confianza al 80% sitúa el precio entre **{pred['ic80_inf']} EUR** y **{pred['ic80_sup']} EUR**.

## Eventos OPA recientes considerados
{bullets_hitos}

## Advertencia
Esta previsión combina información histórica con eventos discretos de la OPA hostil BBVA-Sabadell y no constituye una recomendación de inversión. La cotización puede divergir significativamente del intervalo proyectado ante noticias no anticipadas.

---
*Pipeline: Prophet con holidays OPA · MLflow tracking · src.data_loader cache*
"""
    return informe

print(reporter_agent(resumen_serie, pred))

## Orquestador de los 3 agentes

Función única que el front-end (Gradio del NB 07) puede llamar.

In [ ]:
def orquestar(modelo='prophet'):
    resumen = data_agent()
    path_map = {
        'prophet':           ROOT / 'data' / 'sab_forecast_prophet_90d.csv',
        'pycaret':           ROOT / 'data' / 'sab_forecast_pycaret_90d.csv',
        'pycaret_horizontes':ROOT / 'data' / 'sab_forecast_pycaret_horizontes_90d.csv',
        'regresores':        ROOT / 'data' / 'sab_forecast_regresores_90d.csv',
    }
    path = path_map.get(modelo, path_map['prophet'])
    if not path.exists():
        return f'(falta el CSV {path.name} -- ejecuta el notebook que lo genera primero)'
    pred = forecast_agent(path)
    return reporter_agent(resumen, pred)

print(orquestar('prophet'))

In [ ]:
# Guardar el informe como markdown para el NB 07
out = ROOT / 'data' / 'informe_agente_prophet.md'
out.write_text(orquestar('prophet'), encoding='utf-8')
print(f'Informe guardado en {out}')